# NASA POWER: processamento e visualizacao individual

Este notebook carrega a base NASA POWER, aplica explicitamente o periodo definido para a fonte e gera somente a figura historica individual.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180

def encontrar_pasta_2026(inicio=Path.cwd()):
    inicio = inicio.resolve()
    for pasta in (inicio, *inicio.parents):
        candidata = pasta / '2026'
        if candidata.exists():
            return candidata
    raise FileNotFoundError('Nao foi encontrada a pasta 2026 a partir do diretorio atual.')

PASTA_2026 = encontrar_pasta_2026()
PASTA_SAIDA = PASTA_2026 / 'visualizacoes_comparativas_bases'
PASTA_FONTE = PASTA_SAIDA / 'figuras_por_fonte'
PASTA_FONTE.mkdir(parents=True, exist_ok=True)

FREQUENCIA_VISUAL = 'D'

ROTULOS = {
    'radiacao_onda_curta': 'Radiacao de onda curta (W/m2)',
    'temperatura': 'Temperatura (C)',
    'umidade_relativa': 'Umidade relativa (%)',
}

def recortar_periodo(dados, inicio, fim):
    return dados.loc[
        (dados['data_hora'] >= inicio) & (dados['data_hora'] <= fim)
    ].copy().reset_index(drop=True)

def serie_visual(dados, coluna):
    return dados.set_index('data_hora')[coluna].resample(FREQUENCIA_VISUAL).mean()

def plotar_evolucao(dados, nome, colunas, arquivo, ncol=1):
    colunas = [coluna for coluna in colunas if coluna in dados.columns]
    nlin = int(np.ceil(len(colunas) / ncol))
    fig, eixos = plt.subplots(nlin, ncol, figsize=(14, 3.6 * nlin), sharex=True, squeeze=False)
    for eixo, coluna in zip(eixos.flat, colunas):
        serie_visual(dados, coluna).plot(ax=eixo, color='#1f5a85', linewidth=0.8)
        eixo.set_title(ROTULOS.get(coluna, coluna))
        eixo.set_xlabel('')
        eixo.set_ylabel('')
    for eixo in eixos.flat[len(colunas):]:
        eixo.remove()
    inicio = dados['data_hora'].min().strftime('%d/%m/%Y')
    fim = dados['data_hora'].max().strftime('%d/%m/%Y')
    fig.suptitle(f'Evolucao temporal - {nome} ({inicio} a {fim}; media diaria)', y=1.01)
    fig.tight_layout()
    fig.savefig(arquivo, bbox_inches='tight')
    plt.show()
    plt.close(fig)

PASTA_NASA = PASTA_2026 / 'dados_brutos_nasa_power'
INICIO_NASA = pd.Timestamp('2024-01-05 00:00:00')
FIM_NASA = pd.Timestamp('2026-01-05 23:00:00')
INVALIDOS = [-9999, -999, -6999, 7999, 9999]

def converter_numericas(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors='coerce').replace(INVALIDOS, np.nan)
    return df

def ler_nasa_power():
    partes = []
    for arquivo in sorted(PASTA_NASA.glob('*.csv')):
        linhas = arquivo.read_text(encoding='utf-8').splitlines()
        cabecalho = next(i for i, linha in enumerate(linhas) if linha.startswith('YEAR,'))
        bruto = pd.read_csv(arquivo, skiprows=cabecalho)
        bruto = converter_numericas(bruto, ['ALLSKY_SFC_SW_DWN', 'T2M', 'RH2M'])
        bruto['data_hora'] = pd.to_datetime(
            bruto[['YEAR', 'MO', 'DY', 'HR']].rename(
                columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day', 'HR': 'hour'}
            )
        )
        partes.append(bruto[['data_hora', 'ALLSKY_SFC_SW_DWN', 'T2M', 'RH2M']])
    dados = pd.concat(partes, ignore_index=True).rename(columns={
        'ALLSKY_SFC_SW_DWN': 'radiacao_onda_curta',
        'T2M': 'temperatura',
        'RH2M': 'umidade_relativa',
    })
    dados = dados.sort_values('data_hora').drop_duplicates('data_hora').reset_index(drop=True)
    return recortar_periodo(dados, INICIO_NASA, FIM_NASA)

nasa = ler_nasa_power()
print(f'NASA POWER: {len(nasa)} registros | {nasa.data_hora.min()} a {nasa.data_hora.max()}')
display(nasa.describe(include='all'))


In [ ]:
plotar_evolucao(nasa, 'NASA POWER', ['radiacao_onda_curta', 'temperatura', 'umidade_relativa'], PASTA_FONTE / 'nasa_power_evolucao.png')
